# Denoising experiment — CNN training data preview

Compare **raw**, **denoised**, and **vocals-stem** spectrograms for Abdul Latif Khan - Basant Mukrahi using [vocalprep](https://github.com/jon-myers/denoising-experiments).

**Install** (first time only; downloads ~1 GB of models on first run):

```bash
pip install "vocalprep @ git+https://github.com/jon-myers/denoising-experiments.git#subdirectory=vocalprep"
```

Full-track processing on ~10 min audio may take several minutes on CPU. Stems are cached under `output/denoised/{piece_id}/`.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio, display
from PIL import Image
from idtap import Piece, SwaraClient
from idtap.classes.trajectory import Trajectory

INST = 0
STRING_IDX = 0
SR = 22050
SILENT_TRAJECTORY_ID = 12
SKIP_IDTAP_IDS = {7, 8, 9, 10, 11, 13}
NUM_TRAJECTORIES = 10

MIN_FREQUENCY = 75
MAX_FREQUENCY = 2400
BINS_PER_OCTAVE = 72
HOP_LENGTH = 512
N_BINS = int(np.ceil(BINS_PER_OCTAVE * np.log2(MAX_FREQUENCY / MIN_FREQUENCY)))
CLIP_DURATION = 1.0
CQT_FRAMES_1S = int(np.ceil(CLIP_DURATION * SR / HOP_LENGTH))
EXPORT_IMAGE_WIDTH = CQT_FRAMES_1S * 4

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)
CNN_OUTPUT_DIR = OUTPUT_DIR / "cnn_dataset"

COMPOSITE_LABELS: dict[int, list[int]] = {4: [2, 1], 5: [1, 3]}


@dataclass(frozen=True)
class LabeledSegment:
    label: int | str
    start_frac: float
    end_frac: float


def replace_zeros(data: np.ndarray) -> np.ndarray:
    nonzero = data[np.nonzero(data)]
    if nonzero.size == 0:
        return data
    out = data.copy()
    out[data == 0] = np.min(nonzero)
    return out


def compute_spec_display(y: np.ndarray, sample_rate: int) -> np.ndarray:
    constantq = np.abs(
        librosa.cqt(
            y,
            sr=sample_rate,
            hop_length=HOP_LENGTH,
            fmin=MIN_FREQUENCY,
            n_bins=N_BINS,
            bins_per_octave=BINS_PER_OCTAVE,
        )
    )
    return np.flipud(np.log10(replace_zeros(constantq)))


def dur_fractions(traj: Trajectory, n_segments: int) -> list[float]:
    if traj.dur_array and len(traj.dur_array) >= n_segments:
        return [float(x) for x in traj.dur_array[:n_segments]]
    if n_segments == 2:
        return [1 / 3, 2 / 3]
    return [1.0 / n_segments] * n_segments


def segments_from_fractions(labels: list[int | str], fracs: list[float]) -> list[LabeledSegment]:
    starts = [0.0]
    for frac in fracs:
        starts.append(starts[-1] + frac)
    starts[-1] = 1.0
    return [
        LabeledSegment(label=label, start_frac=starts[i], end_frac=starts[i + 1])
        for i, label in enumerate(labels)
    ]


def iter_labeled_segments(traj: Trajectory) -> list[LabeledSegment]:
    if traj.id in SKIP_IDTAP_IDS:
        return []
    if traj.id == SILENT_TRAJECTORY_ID:
        return [LabeledSegment(label="silent", start_frac=0.0, end_frac=1.0)]
    if traj.id in {0, 1, 2, 3}:
        return [LabeledSegment(label=traj.id, start_frac=0.0, end_frac=1.0)]
    if traj.id in COMPOSITE_LABELS:
        labels = COMPOSITE_LABELS[traj.id]
        return segments_from_fractions(labels, dur_fractions(traj, len(labels)))
    if traj.id == 6:
        n_segments = len(traj.dur_array) if traj.dur_array else max(len(traj.pitches) - 1, 1)
        fracs = dur_fractions(traj, n_segments)
        return segments_from_fractions([1] * n_segments, fracs)
    return []


def trajectory_export_label(traj: Trajectory) -> int | str | None:
    segments = iter_labeled_segments(traj)
    if not segments:
        return None
    return segments[0].label


def load_clip(y_path_or_audio, offset: float, sample_rate: int = SR) -> tuple[np.ndarray, int]:
    y, loaded_sr = librosa.load(
        y_path_or_audio,
        sr=sample_rate,
        mono=True,
        offset=offset,
        duration=CLIP_DURATION,
    )
    target_samples = int(round(CLIP_DURATION * loaded_sr))
    if len(y) < target_samples:
        y = np.pad(y, (0, target_samples - len(y)))
    elif len(y) > target_samples:
        y = y[:target_samples]
    return y, loaded_sr


def build_traj_selections(piece_obj: Piece) -> list[dict[str, object]]:
    trajectories = piece_obj.all_trajectories(inst=INST, string_idx=STRING_IDX)
    start_times = piece_obj.traj_start_times(inst=INST, string_idx=STRING_IDX)
    selections: list[dict[str, object]] = []
    for idx, traj in enumerate(trajectories):
        if idx >= len(start_times):
            break
        traj_start = float(start_times[idx])
        selections.append(
            {"traj": traj, "index": idx, "start": traj_start, "end": traj_start + float(traj.dur_tot)}
        )
    return selections


def _plot_cqt_panel(
    ax,
    *,
    traj: Trajectory,
    y: np.ndarray,
    sample_rate: int,
    title: str,
) -> None:
    segment_duration = len(y) / sample_rate
    spec_display = compute_spec_display(y, sample_rate)
    vmin = float(spec_display.min())
    vmax = float(spec_display.max())
    num_samples = 200
    xs = np.linspace(0, 1, num_samples, endpoint=False)
    contour_times = xs * traj.dur_tot
    contour_freqs = None if traj.id == SILENT_TRAJECTORY_ID else np.array(
        [traj.compute(float(x), log_scale=False) for x in xs]
    )
    img = ax.imshow(
        spec_display,
        aspect="auto",
        origin="upper",
        extent=[0, segment_duration, MIN_FREQUENCY, MAX_FREQUENCY],
        cmap="magma",
        vmin=vmin,
        vmax=vmax,
    )
    if contour_freqs is not None:
        ax.plot(contour_times, contour_freqs, color="cyan", linewidth=2)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Frequency (Hz)")
    ax.set_title(title)
    plt.colorbar(img, ax=ax, label="log10(|CQT|)")


def plot_trajectory_cqt_compare(
    *,
    piece: Piece,
    traj: Trajectory,
    traj_index: int,
    traj_start: float,
    clips: dict[str, tuple[np.ndarray, int]],
) -> None:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
    label = trajectory_export_label(traj)
    fig.suptitle(
        f"{piece.title} — #{traj_index} id={traj.id} ({traj.name_}), "
        f"label={label}, {traj.dur_tot:.2f}s @ {traj_start:.2f}s",
        fontsize=12,
    )
    for ax, (name, (y, sr)) in zip(axes, clips.items()):
        _plot_cqt_panel(ax, traj=traj, y=y, sample_rate=sr, title=name)
    fig.tight_layout()
    plt.show()


def plot_saved_cnn_png(path: Path) -> None:
    if not path.exists():
        print(f"CNN PNG not found: {path}")
        return
    img = Image.open(path)
    fig, ax = plt.subplots(figsize=(6, 8))
    ax.imshow(img)
    ax.set_title(f"Existing CNN PNG: {path.name}")
    ax.axis("off")
    fig.tight_layout()
    plt.show()


def rms(y: np.ndarray) -> float:
    return float(np.sqrt(np.mean(np.square(y))))


def hf_energy_db(y: np.ndarray, sample_rate: int, fmin: float = 8000.0) -> float:
    spec = np.abs(librosa.stft(y, n_fft=2048, hop_length=HOP_LENGTH))
    freqs = librosa.fft_frequencies(sr=sample_rate, n_fft=2048)
    hf_mask = freqs >= fmin
    hf = spec[hf_mask].mean()
    total = spec.mean()
    if total <= 0 or hf <= 0:
        return float("-inf")
    return float(20 * np.log10(hf / total))


def spec_mse(a: np.ndarray, b: np.ndarray, sample_rate: int) -> float:
    sa = compute_spec_display(a, sample_rate)
    sb = compute_spec_display(b, sample_rate)
    n = min(sa.shape[1], sb.shape[1])
    return float(np.mean((sa[:, :n] - sb[:, :n]) ** 2))

In [ ]:
PIECE_ID = "6824de49abc4705438ce918b"
AUDIO_PATH = OUTPUT_DIR / f"Abdul Latif Khan - Basant Mukrahi_{PIECE_ID}.wav"
DENOISE_DIR = OUTPUT_DIR / "denoised" / PIECE_ID
CNN_PIECE_DIR = CNN_OUTPUT_DIR / PIECE_ID

if not AUDIO_PATH.exists():
    raise FileNotFoundError(f"Missing audio: {AUDIO_PATH}")

client = SwaraClient()
piece = Piece.from_json(client.get_piece(PIECE_ID))
traj_selections = build_traj_selections(piece)

print(f"Title: {piece.title}")
print(f"Piece ID: {PIECE_ID}")
print(f"Audio: {AUDIO_PATH}")
print(f"Trajectories: {len(traj_selections)}")
print(f"Existing CNN images: {CNN_PIECE_DIR / 'images'}")

In [ ]:
from vocalprep import VocalPipeline

DENOISE_DIR.mkdir(parents=True, exist_ok=True)
base = AUDIO_PATH.stem
denoised_path = DENOISE_DIR / f"{base}.denoised.wav"
vocals_path = DENOISE_DIR / f"{base}.vocals.wav"

if denoised_path.exists() and vocals_path.exists():
    print(f"Using cached stems in {DENOISE_DIR}")
    result_denoised = denoised_path
    result_vocals = vocals_path
else:
    print("Running vocalprep (denoise + karaoke separation) — first run downloads ~1 GB of models ...")
    pipe = VocalPipeline(output_dir=DENOISE_DIR)
    result = pipe.process(AUDIO_PATH)
    result_denoised = result.denoised
    result_vocals = result.vocals
    print(f"  denoised: {result_denoised}")
    print(f"  vocals:   {result_vocals}")
    if result.noise:
        print(f"  noise:    {result.noise}")
    if result.accompaniment:
        print(f"  accomp:   {result.accompaniment}")

In [ ]:
trajectories = piece.all_trajectories(inst=INST, string_idx=STRING_IDX)
first_nonsilent_idx = next(
    (idx for idx, t in enumerate(trajectories) if t.id != SILENT_TRAJECTORY_ID),
    None,
)
if first_nonsilent_idx is None:
    raise RuntimeError("No non-silent trajectory found.")

preview_selections = [
    sel
    for sel in traj_selections
    if first_nonsilent_idx <= sel["index"] < first_nonsilent_idx + NUM_TRAJECTORIES
]

print(
    f"Previewing {len(preview_selections)} trajectories "
    f"(indices {preview_selections[0]['index']}–{preview_selections[-1]['index']})"
)

for sel in preview_selections:
    traj = sel["traj"]
    label = trajectory_export_label(traj)
    offset = sel["start"]

    y_raw, sr = load_clip(AUDIO_PATH, offset=offset)
    y_denoised, _ = load_clip(result_denoised, offset=offset)
    y_vocals, _ = load_clip(result_vocals, offset=offset)

    print(
        f"\nPlayback — #{sel['index']} id={traj.id} ({traj.name_}), "
        f"label={label}, {CLIP_DURATION}s clip"
    )
    display(Audio(data=y_raw, rate=sr))

    plot_trajectory_cqt_compare(
        piece=piece,
        traj=traj,
        traj_index=sel["index"],
        traj_start=offset,
        clips={
            "Raw mix": (y_raw, sr),
            "Denoised mix": (y_denoised, sr),
            "Vocals stem": (y_vocals, sr),
        },
    )

    cnn_png = CNN_PIECE_DIR / "images" / f"{sel['index']:04d}_{label}.png"
    plot_saved_cnn_png(cnn_png)

In [ ]:
import pandas as pd

metric_rows: list[dict[str, object]] = []

for sel in preview_selections:
    traj = sel["traj"]
    label = trajectory_export_label(traj)
    offset = sel["start"]

    y_raw, sr = load_clip(AUDIO_PATH, offset=offset)
    y_denoised, _ = load_clip(result_denoised, offset=offset)
    y_vocals, _ = load_clip(result_vocals, offset=offset)

    metric_rows.append(
        {
            "traj_index": sel["index"],
            "label": label,
            "rms_raw": rms(y_raw),
            "rms_denoised": rms(y_denoised),
            "rms_vocals": rms(y_vocals),
            "hf_db_raw": hf_energy_db(y_raw, sr),
            "hf_db_denoised": hf_energy_db(y_denoised, sr),
            "hf_db_vocals": hf_energy_db(y_vocals, sr),
            "mse_raw_vs_denoised": spec_mse(y_raw, y_denoised, sr),
            "mse_raw_vs_vocals": spec_mse(y_raw, y_vocals, sr),
        }
    )

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df)

silent_df = metrics_df[metrics_df["label"] == "silent"]
voiced_df = metrics_df[metrics_df["label"] != "silent"]

if not silent_df.empty:
    print("\nSilent trajectories — RMS (lower = cleaner gaps):")
    print(silent_df[["traj_index", "rms_raw", "rms_denoised", "rms_vocals"]].to_string(index=False))

if not voiced_df.empty:
    print("\nVoiced trajectories — HF energy above 8 kHz (dB relative to total):")
    print(
        voiced_df[
            ["traj_index", "label", "hf_db_raw", "hf_db_denoised", "hf_db_vocals"]
        ].to_string(index=False)
    )

print("\nMean spectrogram MSE (raw reference):")
print(f"  raw vs denoised: {metrics_df['mse_raw_vs_denoised'].mean():.4f}")
print(f"  raw vs vocals:   {metrics_df['mse_raw_vs_vocals'].mean():.4f}")